In [35]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# Ruta del archivo
file_path = "OrderAllocation_v2.csv"

# Cargar el CSV en un DataFrame
df = pd.read_csv(file_path)

# Mostrar las primeras filas del DataFrame
df.head()

,orderLine.order.orderIdentifier,orderLine.orderLineNumber,productItem.partNumber,quantityRequired,quantityRequiredUnits,quantityAllocated,quantityAllocatedUnits,allocationComment,alternateItems,sourceLink
0,100044332,100,45000223,56,ea,54,ea,"2020-08-22, 0",NaN,http://lighttree.com/100044332
1,100044332,100,2530020,56,ea,53,ea,"2020-08-20, 25","2530003, 8220072",http://lighttree.com/100044332
2,100044332,100,39000221,56,ea,56,ea,NaN,39000224,http://lighttree.com/100044332
3,100044332,100,29000462,56,ea,56,ea,NaN,29000469,http://lighttree.com/100044332
4,100044332,100,83600300,56,ea,56,ea,NaN,83600200,http://lighttree.com/100044332


Seleccionamos las columnas que queramos analizar y creamos nuevas a partir de ellas

In [36]:
# Separar la columna allocationComment en dos nuevas columnas
df[['allocationDate', 'allocationValue']] = df['allocationComment'].str.split(',', expand=True)

# Convertir allocationDate a formato de fecha y allocationValue a número
df['allocationDate'] = pd.to_datetime(df['allocationDate'], errors='coerce')
df['allocationValue'] = pd.to_numeric(df['allocationValue'], errors='coerce')

Primero lo analizaremos y limpiaremos con NUMPY

In [37]:
# Convertir la columna allocationValue a un array de NumPy
allocation_values_array = df["allocationValue"].to_numpy()

# Calcular Q1, Q3 e IQR en NumPy
Q1 = np.nanpercentile(allocation_values_array, 25)
Q3 = np.nanpercentile(allocation_values_array, 75)
IQR = Q3 - Q1

# Definir límites de outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filtrar valores dentro de los límites para calcular la nueva media sin outliers
filtered_values = allocation_values_array[(allocation_values_array >= lower_bound) & (allocation_values_array <= upper_bound)]
new_mean = np.nanmean(filtered_values)  # Calcular la media ignorando NaN

# Reemplazar valores NaN y outliers con la nueva media usando np.where()
allocation_values_array = np.where(
    (np.isnan(allocation_values_array)) | (allocation_values_array < lower_bound) | (allocation_values_array > upper_bound),
    new_mean,
    allocation_values_array
)

# Asignar el array limpio de vuelta al DataFrame
df["allocationValue"] = allocation_values_array

# Verificar si quedan valores nulos
null_count_final = np.isnan(df["allocationValue"]).sum()

# Mostrar estadísticas finales
df["allocationValue"].describe(), null_count_final

#print(df["allocationValue"])

(count    234.000000
 mean      59.580645
 std       31.793024
 min        0.000000
 25%       59.580645
 50%       59.580645
 75%       59.580645
 max      309.000000
 Name: allocationValue, dtype: float64,
 np.int64(0))

En este apartado haremos lo mismo con PANDAS

In [ ]:
# Convertir la columna allocationValue a un array de tipo float
allocation_values = df['allocationValue'].to_numpy(dtype=float)

# 1. np.isnan(arr) - Detectar valores NaN
nan_mask = np.isnan(allocation_values)  # Devuelve un array booleano donde hay NaN
#print("¿Dónde hay valores NaN?:", nan_mask)

# Contar cuántos valores nulos hay
num_nulls = np.sum(nan_mask)

#print("\nCantidad de valores nulos en la columna Allocation_Values:", num_nulls)

allocation_stats = {
    "mean": df["allocationValue"].mean(),
    "median": df["allocationValue"].median(),
    "min": df["allocationValue"].min(),
    "max": df["allocationValue"].max(),
    "std": df["allocationValue"].std()
}

print(allocation_stats)

# Calcular Q1 y Q3
Q1 = df["allocationValue"].quantile(0.25)
Q3 = df["allocationValue"].quantile(0.75)

# Calcular IQR
IQR = Q3 - Q1

# Definir límites para outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filtrar valores dentro de los límites
df_filtered = df[(df["allocationValue"] >= lower_bound) & (df["allocationValue"] <= upper_bound)]

# Recalcular estadísticas después de eliminar outliers
allocation_stats_filtered = {
    "mean": df_filtered["allocationValue"].mean(),
    "median": df_filtered["allocationValue"].median(),
    "min": df_filtered["allocationValue"].min(),
    "max": df_filtered["allocationValue"].max(),
    "std": df_filtered["allocationValue"].std()
}

print(allocation_stats_filtered)

# Rellenar valores nulos en allocationValue con la nueva media calculada
df_filtered.loc[:, "allocationValue"].fillna(allocation_stats_filtered["mean"], inplace=True)

# Verificar que ya no haya valores nulos en la columna
null_count_after_fill = df_filtered["allocationValue"].isnull().sum()

print(df_filtered["allocationValue"], null_count_after_fill)

df["allocationValue"] = df_filtered["allocationValue"]

Creamos una nueva columna para sacar la diferencia entre "quantityRequiered" y "quantityAllocated"

Tambien creamos y mostramos nuevas columnas que se añadiran al dataset final

In [ ]:
# Crear la columna allocationDifference
df_filtered.loc[:, "allocationDifference"] = df_filtered["quantityRequired"] - df_filtered["quantityAllocated"]

# Extraer información de allocationDate
df_filtered.loc[:, "allocationYear"] = df_filtered["allocationDate"].dt.year
df_filtered.loc[:, "allocationMonth"] = df_filtered["allocationDate"].dt.month
df_filtered.loc[:, "allocationWeekday"] = df_filtered["allocationDate"].dt.day_name()

# Mostrar algunas filas con las nuevas columnas
df_filtered[["quantityRequired", "quantityAllocated", "allocationDifference", 
             "allocationDate", "allocationYear", "allocationMonth", "allocationWeekday"]].head()

Vamos con Scikit-Learn

In [ ]:
# Convertir la columna allocationValue a un array de NumPy (scikit-learn requiere matriz 2D)
allocation_values_array = df["allocationValue"].to_numpy().reshape(-1, 1)

# Calcular Q1, Q3 e IQR en NumPy
Q1 = np.nanpercentile(allocation_values_array, 25)
Q3 = np.nanpercentile(allocation_values_array, 75)
IQR = Q3 - Q1

# Definir límites de outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Reemplazar outliers con NaN
allocation_values_array[(allocation_values_array < lower_bound) | (allocation_values_array > upper_bound)] = np.nan

# Crear el imputador para reemplazar NaN con la media
imputer = SimpleImputer(strategy="mean")

# Aplicar la transformación a los datos
allocation_values_imputed = imputer.fit_transform(allocation_values_array)

# Asignar los valores limpios de vuelta al DataFrame
df["allocationValue"] = allocation_values_imputed.flatten()

# Verificar si quedan valores nulos
null_count_final = df["allocationValue"].isnull().sum()

# Mostrar estadísticas finales
df["allocationValue"].describe(), null_count_final